# AB-200 Questions: Q21–30 — Python Syntax and Fundamentals

Part of a series — final notebook of the set. See [Q1–10](Q-1-10-solutions.ipynb) and [Q11–20](Q-11-20-solutions.ipynb) for the earlier ones.

For each question we follow the same drill, in this order:

1. **Original question** — quoted as written, with its hint, so the notebook stands on its own.
2. **Restate the problem** — in plain language, so we're sure we're solving the right thing.
3. **Think algorithmically** — write the steps as an ordinary person would describe them, with no loops/ifs/syntax.
4. **Brute force** — the first correct idea that comes to mind, even if it's wasteful.
5. **Optimal** — built *on top of* the brute force by asking "what work are we repeating, and can we avoid it?"
6. **Tests** — normal case, edge cases, and asserts that actually run.
7. **Interview traps** — the specific mistakes that make candidates lose points on this exact question.

---
## Q21. List Subset Check

> **List Subset Check:** Check if one list is a subset of another (if all elements of the smaller list are contained in the larger list).
> _Hint:_ Convert the larger list to a set and ensure every element of the smaller list is in that set.

**Restate:** Given two lists, decide whether every *distinct value* in the first appears somewhere in the second (duplicate counts in either list don't matter — same "distinct values" convention as Q20's intersection, stated explicitly since the prompt doesn't).

**Algorithmic thinking (no syntax):**
1. Walk through every element of the candidate subset list.
2. For each one, check whether it also appears somewhere in the other list.
3. The moment you find an element that *doesn't* appear in the other list, you know the answer is no — stop immediately.
4. If every element passes the check, the answer is yes.

This is Q20's exact membership-check pattern, just with the loop's exit condition flipped: Q20 collected everything that *matched*; this one bails out the instant something *doesn't*.

In [ ]:
# Brute force — membership check against a list. O(n * m) time.

def is_subset_brute(smaller, larger):
    for item in smaller:
        if item not in larger:
            return False
    return True

print(is_subset_brute([1, 2], [1, 2, 3, 4]))   # True
print(is_subset_brute([1, 5], [1, 2, 3, 4]))   # False

**Optimal:** Build the `set` of the larger list once, up front, exactly as in Q20. Every membership check afterward is O(1) average, taking the total from O(n·m) to O(n + m).

In [ ]:
# Optimal — set built once, O(1) average membership checks.
# O(n + m) time.

def is_subset_optimal(smaller, larger):
    larger_set = set(larger)
    return all(item in larger_set for item in smaller)

print(is_subset_optimal([1, 2], [1, 2, 3, 4]))   # True
print(is_subset_optimal([1, 5], [1, 2, 3, 4]))   # False

# Python's set type has this built in directly:
def is_subset_native(smaller, larger):
    return set(smaller) <= set(larger)   # or set(smaller).issubset(larger)

print(is_subset_native([1, 2], [1, 2, 3, 4]))    # True

In [ ]:
# Tests

assert is_subset_optimal([], [1, 2, 3]) is True             # edge: empty set is a subset of anything
assert is_subset_optimal([], []) is True                     # edge: empty subset of empty
assert is_subset_optimal([1, 2, 3], []) is False              # edge: nothing is a subset of empty except empty itself
assert is_subset_optimal([1, 2], [1, 2, 3, 4]) is True
assert is_subset_optimal([1, 5], [1, 2, 3, 4]) is False
assert is_subset_optimal([1, 2, 3], [1, 2, 3]) is True        # edge: identical lists
assert is_subset_optimal([1, 1, 2], [1, 2, 3]) is True        # duplicates in the candidate don't matter

for smaller, larger in [([], [1, 2, 3]), ([1, 2], [1, 2, 3, 4]), ([1, 5], [1, 2, 3, 4]), ([1, 2, 3], [])]:
    assert is_subset_brute(smaller, larger) == is_subset_optimal(smaller, larger) == is_subset_native(smaller, larger)

print("All Q21 tests passed.")

**Interview traps:**
- **The empty list is a subset of *everything*, including another empty list — but nothing (except empty itself) is a subset of empty.** This is the classic "vacuous truth" edge case: `all(...)` over an empty iterable is `True` by definition (there's no element to violate the condition), so `is_subset_optimal([], anything)` returns `True` automatically. Trace through *why* rather than being surprised when this passes.
- **Direction matters and is easy to get backwards.** "Is `A` a subset of `B`" is not symmetric — swapping the arguments asks a different question. State clearly which list is the candidate and which is the container, both in your restatement and in your function's parameter names (`smaller, larger` here, not `a, b`).
- **`set(smaller) <= set(larger)` reads naturally but relies on knowing sets support comparison operators for subset testing** — `<` for *strict* subset (subset, but not equal), `<=` for subset-or-equal. Mixing these up gives a subtly wrong answer for identical lists (`<` would say `[1,2,3]` is not a subset of `[1,2,3]`, which fails our "identical lists" test above).
- **Hashability requirement, once again** — a list of unhashable elements can't be converted to a `set` at all, forcing the O(n·m) brute-force fallback. This is now the fourth question in a row (Q12, Q19, Q20, Q21) where the same constraint shows up; naming that pattern explicitly is a strong signal that you're generalizing, not just pattern-matching each question in isolation.

---
## Q22. Sort Words by Length

> **Sort Words by Length:** Sort a list of words by their length (shortest to longest).
> _Hint:_ Use `sorted(words, key=len)` to sort by length of each word.

**Restate:** Given a list of words, reorder them ascending by character count, keeping words of equal length in their original relative order (same stability convention as Q16).

**Algorithmic thinking (no syntax):** Identical shape to Q16 — "sort by some derived value" — except the value we're comparing by is each word's length instead of a tuple's second element. If you solved Q16, you already have the pseudocode: sweep and swap adjacent items whose *lengths* are out of order, repeat until settled.

Because this is the same underlying idea as Q16, we'll move faster here and spend the real attention on what's different: `len()` on a string counts *Unicode code points*, not bytes or "visual characters" — which becomes the interesting edge case below.

In [ ]:
# Brute force — bubble sort, comparing by len() explicitly. O(n^2) time.

def sort_by_length_brute(words):
    result = list(words)
    n = len(result)
    for i in range(n):
        for j in range(n - 1 - i):
            if len(result[j]) > len(result[j + 1]):
                result[j], result[j + 1] = result[j + 1], result[j]
    return result

print(sort_by_length_brute(["banana", "kiwi", "fig", "apple"]))
# ['fig', 'kiwi', 'apple', 'banana']

**Optimal:** `sorted(words, key=len)` — O(n log n), stable, and `len` itself needs no wrapping in a `lambda`: it's already a one-argument function that takes a string and returns its length, which is exactly what `key` wants.

In [ ]:
# Optimal — pass len directly as the key. O(n log n) time.

def sort_by_length_optimal(words):
    return sorted(words, key=len)

print(sort_by_length_optimal(["banana", "kiwi", "fig", "apple"]))
# ['fig', 'kiwi', 'apple', 'banana']

In [ ]:
# Tests

assert sort_by_length_optimal([]) == []                                       # edge: empty list
assert sort_by_length_optimal(["a"]) == ["a"]                                  # edge: single word
assert sort_by_length_optimal(["cat", "dog", "ox"]) == ["ox", "cat", "dog"]     # tie: original order preserved
assert sort_by_length_optimal(["banana", "kiwi", "fig", "apple"]) == ["fig", "kiwi", "apple", "banana"]
assert sort_by_length_optimal([""]) == [""]                                    # edge: empty string is length 0

for words in ([], ["a"], ["cat", "dog", "ox"], ["banana", "kiwi", "fig", "apple"]):
    assert sort_by_length_brute(words) == sort_by_length_optimal(words)

print("All Q22 tests passed.")

In [ ]:
# The trap, demonstrated: len() counts code points, not the single glyph
# a person would see on screen -- same underlying issue as Q2's reversal
# trap, showing up here as a length miscount instead. Built with explicit
# \u escapes so there's no ambiguity about which Unicode form is which.

precomposed = "caf" + "é"        # the accented e is ONE code point (U+00E9)
decomposed = "cafe" + "́"        # plain 'e' + a combining acute accent (U+0301)

print(repr(precomposed), "->", len(precomposed))   # 'café' -> 4
print(repr(decomposed), "->", len(decomposed))      # 'café' -> 5, though it renders identically

**Interview traps:**
- **`sorted(words, key=len)` sorts by *code point count*, which is usually — but not always — what a human means by "word length."** The `café` example above shows two strings that render identically but have different `len()` values; this is rarely the crux of the question, but naming it unprompted (as we did in Q2) signals real Unicode awareness rather than a lucky guess on ASCII-only test cases.
- **`key=len` vs `key=lambda w: len(w)` are functionally identical — pass the function directly when you can.** Wrapping a function in a `lambda` just to call it with one argument and return the result is pure overhead with no behavior change; it's a habit worth dropping once you notice you're doing it.
- **Stability lets you chain sort keys for free.** If a follow-up asks for "shortest to longest, and alphabetically for ties," the answer is `key=lambda w: (len(w), w)` — a tuple key, compared element-by-element, exactly like Q16's stability discussion.
- **Don't reach for a hand-rolled bubble sort as your real answer.** As in Q16, if you write it, immediately flag that you know `sorted()` exists and is asymptotically and practically better — the bubble sort is worth writing once to prove you understand the underlying comparison-and-swap idea, not as your final offer.

---
## Q23. Sort Numeric Strings

> **Sort Numeric Strings:** You have a list of numbers in string form. Sort this list by numeric value (e.g., "2" comes before "11" because 2 < 11).
> _Hint:_ Use `sorted(strings, key=int)` to convert strings to integers for comparison during sort.

**Restate:** Given a list of strings that each represent an integer, sort the list so that the strings appear in ascending order *of the number they represent* — not in the order plain text comparison would produce.

**Algorithmic thinking (no syntax):**
1. For each string in the list, figure out what number it represents.
2. Sort the strings using those numbers to decide order, not the text itself.
3. The output is still a list of the original strings — just reordered.

The phrase "not the text itself" in step 2 is the entire question. Sorting the *strings* directly compares them character by character — `"11"` vs `"2"` compares `'1'` to `'2'` at the first character and decides `"11"` comes first, which is numerically backwards. This question exists specifically to test whether you'll sort text as if it were numbers, which is the natural (wrong) first instinct.

In [ ]:
# "Brute force" here is actually a demonstration of the WRONG naive answer:
# sorting the strings directly, lexicographically. This is what you get
# if you call sorted() on the strings without a key at all.

def sort_numeric_strings_wrong(strings):
    return sorted(strings)   # plain lexicographic (text) sort -- the trap

print(sort_numeric_strings_wrong(["2", "11", "1", "20"]))
# ['1', '11', '2', '20'] -- text order, NOT numeric order

**Optimal (and correct):** Supply `key=int` — each string is converted to an integer *for comparison purposes only*, while the output still contains the original strings. Same O(n log n) sort as the wrong version above; the only change is *what's being compared*, which is the whole lesson of this question.

In [ ]:
# Optimal — key=int. O(n log n) time, same as the wrong version, but
# comparing the right thing.

def sort_numeric_strings_optimal(strings):
    return sorted(strings, key=int)

print(sort_numeric_strings_optimal(["2", "11", "1", "20"]))
# ['1', '2', '11', '20'] -- numeric order, as intended

In [ ]:
# Tests

assert sort_numeric_strings_optimal([]) == []                                       # edge: empty list
assert sort_numeric_strings_optimal(["5"]) == ["5"]                                  # edge: single element
assert sort_numeric_strings_optimal(["2", "11", "1", "20"]) == ["1", "2", "11", "20"]
assert sort_numeric_strings_optimal(["-5", "3", "-1"]) == ["-5", "-1", "3"]           # negative numbers
assert sort_numeric_strings_optimal(["007", "7", "70"]) == ["007", "7", "70"]         # tie: "007" == "7" numerically, order preserved
assert sort_numeric_strings_optimal(["10", "9", "10"]) == ["9", "10", "10"]           # duplicate numeric values kept

try:
    sort_numeric_strings_optimal(["3", "abc", "1"])
    assert False, "expected ValueError for a non-numeric string"
except ValueError:
    pass   # edge: non-numeric input correctly rejected by int()

# Confirm the "wrong" version really is wrong -- it disagrees with the
# correct one on an input where text order and numeric order differ.
assert sort_numeric_strings_wrong(["2", "11", "1", "20"]) != sort_numeric_strings_optimal(["2", "11", "1", "20"])

print("All Q23 tests passed.")

**Interview traps:**
- **`sorted(strings)` with no key is the entire trap, and it doesn't error — it just silently gives the wrong order.** There's no exception to catch you here; the only defense is recognizing that comparing strings numerically requires an explicit numeric key. This is the most "looks right until you check it" question in the set.
- **`"007"` and `"7"` are numerically equal but textually different** — this is a legitimate tie in the numeric ordering, and Python's sort stability determines which comes first (whichever appeared first in the input). Confirm this is the behavior you want, since it's easy to assume incorrectly that leading zeros would somehow be normalized away in the output (they aren't — the *strings* are untouched; only the *comparison* uses the numeric value).
- **Non-numeric strings crash `int()` with `ValueError`, not silently sort to one end.** If the input isn't guaranteed to be clean numeric strings, decide explicitly how to handle bad entries (skip them? raise? sort them last?) rather than letting an uncaught exception be your answer to "what if the input is dirty."
- **Negative numbers as strings work correctly with `key=int`** (`int("-5")` parses fine) but would sort very wrong under plain lexicographic order (`"-5"` starts with `'-'`, which compares oddly against digit characters) — another reason the naive `sorted(strings)` answer fails silently rather than obviously.
- **This is functionally the reverse problem of Q18** (worked with a real integer, converted to text) — here you start with text and need the *numeric* interpretation for comparison only, while the final output stays textual. Recognizing when a problem wants you to convert types just for comparison, versus for the actual output, is a recurring interview skill.

---
## Q24. Comprehension – Squares of Evens

> **Comprehension – Squares of Evens:** Generate a list of the squares of all even numbers from 1 to N.
> _Hint:_ Use a list comprehension: `[x*x for x in range(1, N+1) if x % 2 == 0]`.

**Restate:** Given `N`, produce a list of `x*x` for every even `x` from `1` to `N` inclusive, in ascending order of `x`.

**Algorithmic thinking (no syntax):**
1. Walk through every whole number from `1` up to `N`.
2. For each one, check whether it's even.
3. If it is, square it and add that to the result.
4. If it isn't, skip it — move on to the next number.

Here's the detail worth noticing before writing any code: step 2 asks a question ("is this even?") for *every single number*, including the odd ones we're about to throw away. But we already know exactly where the even numbers are — they're every second number, starting at 2. Do we need to ask the question at all?

In [ ]:
# Brute force — walk every number from 1 to N, filter with a modulo
# check. O(N) time, but every single number (even the ones we discard)
# costs one iteration AND one modulo check.

def squares_of_evens_brute(N):
    result = []
    for x in range(1, N + 1):
        if x % 2 == 0:
            result.append(x * x)
    return result

print(squares_of_evens_brute(10))   # [4, 16, 36, 64, 100]

**Optimal:** Step directly through only the even numbers — `range(2, N + 1, 2)` — so there's no modulo check and no wasted iteration on odd numbers at all. Still O(N) overall (technically O(N/2), which is the same order of growth), but it does roughly half the work of the brute force and asks zero unnecessary questions. Expressed as a one-line comprehension, since that's the idiom the hint itself points at.

In [ ]:
# Optimal — step by 2, starting at the first even number. No modulo
# check, no wasted iterations on odd numbers.

def squares_of_evens_optimal(N):
    return [x * x for x in range(2, N + 1, 2)]

print(squares_of_evens_optimal(10))   # [4, 16, 36, 64, 100]

In [ ]:
# Tests

assert squares_of_evens_optimal(0) == []                     # edge: N below the range entirely
assert squares_of_evens_optimal(1) == []                      # edge: N=1, no even numbers in [1, 1]
assert squares_of_evens_optimal(2) == [4]                     # edge: N is itself the first even number
assert squares_of_evens_optimal(-5) == []                     # edge: negative N
assert squares_of_evens_optimal(10) == [4, 16, 36, 64, 100]
assert squares_of_evens_optimal(11) == squares_of_evens_optimal(10)   # odd N: same result as N-1

for N in range(-3, 30):
    assert squares_of_evens_brute(N) == squares_of_evens_optimal(N), N

print("All Q24 tests passed.")

**Interview traps:**
- **"1 to N" is inclusive of `N`.** `range(1, N + 1)` and `range(2, N + 1, 2)` both need the `+ 1` — drop it and you silently lose `N` itself whenever `N` is even, which is exactly the kind of off-by-one that a quick test with an even `N` catches immediately (worth choosing your test cases with that in mind).
- **`range(2, N + 1, 2)` for negative or tiny `N` doesn't need a special case — it just produces an empty range naturally.** `range(2, -4, 2)` and `range(2, 1, 2)` both iterate zero times without raising anything. Trust `range`'s own bounds-handling rather than adding a redundant `if N < 2: return []` guard.
- **This is the point in the set where "the optimization is real but small" is worth being honest about.** Skipping the modulo check roughly halves the work, but both versions are O(N) — don't oversell a constant-factor improvement as an asymptotic one. Precision about *what kind* of improvement you're offering is itself a signal.
- **`x*x` vs `x**2`** — identical result, but `x*x` avoids the (tiny) overhead of Python's general-purpose power operator for the specific case of squaring. Not something to obsess over, but a legitimate thing to mention if asked to micro-optimize further.

---
## Q25. Dictionary Comprehension

> **Dictionary Comprehension:** Given a dictionary `mappings`, invert it to get a new dictionary where values map to keys.
> _Hint:_ Use a dict comprehension: `{v: k for k, v in mappings.items()}`.

**Restate:** Given a dictionary, build a new dictionary where each original *value* becomes a key, and each original *key* becomes that key's value. This only produces a clean, lossless result if the original values are both **unique** and **hashable** — we'll come back to what happens when they aren't.

**Algorithmic thinking (no syntax):**
1. Start a new, empty mapping.
2. Look at each key-value pair in the original, one at a time.
3. In the new mapping, record an entry where the *value* is now the key, and the *key* is now the value.
4. Once every pair has been processed, the new mapping is the answer.

Like Q17 and Q9, there's no brute-force-vs-optimal complexity split here — both versions below visit each pair exactly once, O(n). What's actually interesting about this question is a correctness question hiding behind a syntax question: what happens to step 3 if two different keys shared the same value?

In [ ]:
# "Brute force" — explicit loop, manually assigning into a new dict.
# O(n) time. Functionally identical to the comprehension version;
# the only difference is verbosity.

def invert_dict_brute(mappings):
    inverted = {}
    for key, value in mappings.items():
        inverted[value] = key
    return inverted

print(invert_dict_brute({"a": 1, "b": 2, "c": 3}))   # {1: 'a', 2: 'b', 3: 'c'}

**Optimal:** The dict comprehension from the hint — same O(n) time and the exact same behavior as the loop, just the idiomatic one-line spelling of "build a new dict from this iterable of pairs."

In [ ]:
# Optimal — dict comprehension. O(n) time.

def invert_dict_optimal(mappings):
    return {value: key for key, value in mappings.items()}

print(invert_dict_optimal({"a": 1, "b": 2, "c": 3}))   # {1: 'a', 2: 'b', 3: 'c'}

In [ ]:
# The correctness question, demonstrated: what happens when two keys
# share the same value? Whichever key was processed LAST wins -- the
# earlier one is silently overwritten and lost.

lossy = {"a": 1, "b": 2, "c": 1}   # 'a' and 'c' both map to 1
print(invert_dict_optimal(lossy))   # {1: 'c', 2: 'b'} -- 'a' is gone, not an error

In [ ]:
# Tests

assert invert_dict_optimal({}) == {}                                     # edge: empty dict
assert invert_dict_optimal({"a": 1}) == {1: "a"}                          # edge: single entry
assert invert_dict_optimal({"a": 1, "b": 2, "c": 3}) == {1: "a", 2: "b", 3: "c"}
assert invert_dict_optimal({"a": 1, "b": 2, "c": 1}) == {1: "c", 2: "b"}   # last key wins on collision
assert invert_dict_optimal({"x": "y"}) == {"y": "x"}                      # non-numeric values work fine

for mappings in ({}, {"a": 1}, {"a": 1, "b": 2, "c": 3}, {"a": 1, "b": 2, "c": 1}):
    assert invert_dict_brute(mappings) == invert_dict_optimal(mappings)

try:
    invert_dict_optimal({"a": [1, 2]})   # a list value can't become a dict key
    assert False, "expected TypeError for an unhashable value"
except TypeError:
    pass   # edge: unhashable value correctly rejected

print("All Q25 tests passed.")

**Interview traps:**
- **Silent data loss on duplicate values is the whole question.** The comprehension doesn't error, doesn't warn — it just quietly keeps the last key it saw for each value and drops the rest. If a follow-up asks "what if values aren't unique," the correct answer is "some keys get lost, specifically all but the last one for each repeated value" — and if the requirement is to keep *all* of them, you need a different output shape entirely (a dict mapping each value to a *list* of keys, not a single key).
- **"Last one wins" specifically means last in iteration order**, which for a regular dict is insertion order (Python 3.7+) — so the outcome is deterministic and traceable, not random, even though it might look arbitrary at first glance.
- **Values must be hashable to become keys.** A dictionary with list or dict values can't be inverted this way at all — `{value: key ...}` raises `TypeError: unhashable type` the moment it hits such a value. This is the same hashability constraint that's shown up in every set-based question in this set, just surfacing through dict keys instead.
- **Keys must be unique going in, but that's guaranteed for free** — a `dict` can't have duplicate keys in the first place, so the *original* mapping never has this problem; it's only the *values*, once they become keys in the output, where uniqueness has to be earned rather than assumed.

---
## Q26. Longest Word in Sentence

> **Longest Word in Sentence:** Find the longest word in a sentence and report its length.
> _Hint:_ Split the sentence by spaces and use `max(words, key=len)` to find the longest word (track its length as needed).

**Restate:** Given a sentence, split it into words and find the one with the most characters, along with its length. If multiple words tie for longest, report the *first* one that appears (a specific tie-break convention worth stating explicitly, since the prompt doesn't).

**Algorithmic thinking (no syntax):**
1. Break the sentence into words.
2. Keep track of the longest word seen so far (and its length), starting from the first word.
3. Walk the remaining words one at a time; whenever you find one strictly longer than the current longest, replace your tracked answer with it.
4. Once you've walked every word, whatever you're tracking is the answer.

Step 3's "strictly longer" (not "longer or equal") is what produces the "first one wins on a tie" behavior — an equally-long later word never replaces an earlier one, because it's never *strictly greater*.

In [ ]:
# Brute force — sort all the words by length just to grab the first one.
# Same wasted-full-sort-for-a-single-answer pattern as Q13's "sort then
# index" approach. O(n log n) time to answer a question that doesn't
# need anything sorted, only the single biggest value.

def longest_word_brute(sentence):
    words = sentence.split()
    words_by_length = sorted(words, key=len, reverse=True)
    longest = words_by_length[0]
    return longest, len(longest)

print(longest_word_brute("the quick brown fox jumps"))   # ('quick', 5)

**Optimal:** `max(words, key=len)` — a single O(n) pass that tracks the running best without ever sorting anything. Python's `max()` explicitly guarantees "first maximal item wins" on ties, which is exactly the convention we stated in the restatement — not a coincidence, but worth confirming rather than assuming.

In [ ]:
# Optimal — single pass via max(). O(n) time, O(1) extra space.

def longest_word_optimal(sentence):
    words = sentence.split()
    if not words:
        raise ValueError("sentence has no words")
    longest = max(words, key=len)
    return longest, len(longest)

print(longest_word_optimal("the quick brown fox jumps"))   # ('quick', 5)

In [ ]:
# Tests

assert longest_word_optimal("hello") == ("hello", 5)                              # edge: single word
assert longest_word_optimal("the quick brown fox jumps") == ("quick", 5)
assert longest_word_optimal("a bb ccc") == ("ccc", 3)                              # strictly increasing lengths
assert longest_word_optimal("aa bb cc") == ("aa", 2)                               # edge: three-way tie, first wins
assert longest_word_optimal("Hello, world!") == ("Hello,", 6)                      # punctuation counted, by convention here

for bad_input in ("", "   "):
    try:
        longest_word_optimal(bad_input)
        assert False, f"expected ValueError for {bad_input!r}"
    except ValueError:
        pass   # edge: empty / whitespace-only sentence correctly rejected

for sentence in ("hello", "the quick brown fox jumps", "a bb ccc", "aa bb cc"):
    assert longest_word_brute(sentence) == longest_word_optimal(sentence)

print("All Q26 tests passed.")

**Interview traps:**
- **`max()` on an empty sequence raises `ValueError`, not a graceful `None`.** An empty sentence (or one that's only whitespace, which `.split()` also reduces to an empty list) needs an explicit guard before calling `max()` — letting it crash uncaught is a common miss on this exact question.
- **Punctuation attached to a word inflates its counted length**, exactly as in Q10's word-frequency question. `"Hello,"` is 6 characters including the comma; whether that's the "right" answer depends on whether the question wants word *tokens* (split on whitespace, punctuation included) or word *content* (punctuation stripped first). State which one you're doing — we chose tokens-as-is here since the hint's `.split()` doesn't strip anything.
- **Tie-breaking is a real design decision, not an afterthought.** `max()`'s "first maximal item" behavior is a documented guarantee, not an accident — but if the question wants the *last* longest word, or *all* of the tied longest words, you need different code entirely (e.g., a single pass building a list of ties, or `max()` combined with a second filtering pass).
- **Don't sort just to find one extreme value.** This is the second time in the set (after Q13) that "sort the whole thing, take one end of it" shows up as the tempting-but-wasteful answer — recognizing the pattern (you only need the single best, not a full ordering) is worth calling out explicitly rather than re-deriving it from scratch each time.

---
## Q27. Flatten Nested List (Shallow)

> **Flatten Nested List (Shallow):** Flatten a nested list by one level. For example, `[[1,2],[3,4]]` becomes `[1,2,3,4]`.
> _Hint:_ Use a nested loop or a list comprehension like `[item for sublist in lists for item in sublist]`.

**Restate:** Given a list of lists, produce a single list containing every element from every sublist, in order — but only unwrap *one* level of nesting. If a sublist itself contains a list (`[[1, [2, 3]], [4]]`), that inner list stays intact (`[1, [2, 3], 4]`) rather than being unwrapped too — "shallow" is doing real work in the question's title.

**Algorithmic thinking (no syntax):**
1. Start a new, empty list.
2. Walk through each sublist in the outer list, one at a time.
3. For each sublist, walk through *its* elements, one at a time, placing each directly onto the result list.
4. Once every sublist has been fully walked, the result is the answer.

This is a nested walk, not a nested *search* — there's no nested-loop nested-loop O(n²) trap lurking in the algorithm itself, since every element is visited exactly once regardless of how you write the loops. The real trap here is different, and it's in how you're tempted to *express* the idea in Python — see below.

In [ ]:
# Brute force — the "elegant-looking" trap: sum(lists, []) concatenates
# every sublist onto a growing result, one at a time. Each += copies
# everything accumulated so far -- the exact same repeated-copying
# problem as Q2's string concatenation, just one type up. For k
# sublists, this is O(k^2) in the number of sublists.

def flatten_brute(lists):
    return sum(lists, [])   # looks clean, quietly quadratic

print(flatten_brute([[1, 2], [3, 4], [5]]))   # [1, 2, 3, 4, 5]

**Optimal:** A nested list comprehension — one outer loop over sublists, one inner loop over each sublist's elements, both feeding a single result built in one pass. O(n) time in the total number of elements, with no repeated copying.

In [ ]:
from itertools import chain

# Optimal — nested comprehension, one pass. O(n) time.

def flatten_optimal(lists):
    return [item for sublist in lists for item in sublist]

print(flatten_optimal([[1, 2], [3, 4], [5]]))   # [1, 2, 3, 4, 5]

# itertools.chain.from_iterable does the same thing, lazily -- it
# returns an iterator instead of building the full list immediately,
# which matters if you only need to iterate the flattened result once.
def flatten_chain(lists):
    return list(chain.from_iterable(lists))

print(flatten_chain([[1, 2], [3, 4], [5]]))     # [1, 2, 3, 4, 5]

In [ ]:
# Tests

assert flatten_optimal([]) == []                                    # edge: no sublists at all
assert flatten_optimal([[]]) == []                                   # edge: one empty sublist
assert flatten_optimal([[], [], []]) == []                           # edge: several empty sublists
assert flatten_optimal([[1, 2], [3, 4]]) == [1, 2, 3, 4]
assert flatten_optimal([[1], [], [2, 3], []]) == [1, 2, 3]            # mixed empty and non-empty
assert flatten_optimal([[1, [2, 3]], [4]]) == [1, [2, 3], 4]          # shallow only: inner list NOT unwrapped

for lists in ([], [[]], [[], [], []], [[1, 2], [3, 4]], [[1], [], [2, 3], []]):
    assert flatten_brute(lists) == flatten_optimal(lists) == flatten_chain(lists)

print("All Q27 tests passed.")

In [ ]:
import time

# Let's feel the quadratic blowup, the same way we did for naive
# Fibonacci in Q5 -- lots of small sublists is the worst case for
# sum(lists, []), since it maximizes how many times early elements
# get re-copied.

many_small_lists = [[i] for i in range(20000)]   # 20,000 sublists, one element each

start = time.time()
flatten_brute(many_small_lists)
print(f"flatten_brute:   {time.time() - start:.3f}s")

start = time.time()
flatten_optimal(many_small_lists)
print(f"flatten_optimal: {time.time() - start:.3f}s")

**Interview traps:**
- **`sum(lists, [])` is the single most common "clever-looking but wrong" answer to this exact question**, and the timing above shows why it matters, not just that it's theoretically worse. If you offer it, immediately flag the quadratic behavior — same expectation as the string-concatenation trap in Q2, just with lists instead of strings.
- **"Shallow" means exactly one level, and it's easy to accidentally write a *deep* flatten instead.** `[[1, [2, 3]], [4]]` flattened shallowly stays `[1, [2, 3], 4]` — the inner `[2, 3]` is untouched. A recursive flatten-everything solution silently answers a different (also common, but not this) question; make sure your solution doesn't recurse into sublists-of-sublists unless that's actually what's being asked.
- **`itertools.chain.from_iterable` is lazy** — it returns an iterator, not a list, so `flatten_chain(lists)` without wrapping in `list(...)` doesn't give you a usable list, just an iterator object. This matters if code downstream expects to index into the result or check its length.
- **Both the comprehension and `chain.from_iterable` assume every element of the outer list is itself iterable** (a list, tuple, string, etc.). If the outer list mixes iterables and non-iterables (`[[1, 2], 3]`), the inner loop crashes with `TypeError: 'int' object is not iterable` on the `3` — worth naming as an input assumption if it's not guaranteed by the problem.

---
## Q28. Cumulative Sum

> **Cumulative Sum:** Given a list of numbers, produce a new list where each element is the cumulative sum up to that index of the original list.
> _Hint:_ Loop through the list, accumulating the sum, or use itertools.accumulate for a Pythonic solution.

**Restate:** Given a list of numbers, produce a new list of the same length where each position holds the sum of every number in the original list *up to and including* that position. So `[1, 2, 3, 4]` becomes `[1, 3, 6, 10]` — position 0 is just the first number, position 1 is the first two added together, and so on.

**Algorithmic thinking (no syntax):**
1. Keep a running total, starting at zero.
2. Walk through the original list, one number at a time.
3. Add the current number to the running total.
4. Place the *updated* running total onto the result — not the original number, the total *so far*.
5. Once you've walked the whole list, the result is the answer.

Step 3 and step 4 together are the entire idea: each output position depends on the running total *carried over* from the previous position, plus the one new number. That carry-over is exactly the thing worth not throwing away and recomputing from scratch.

In [ ]:
# Brute force — recompute the sum from scratch at every index, ignoring
# that most of the work was already done for the previous index.
# Summing a slice of length i costs O(i), and we do this for every i
# from 0 to n-1: O(n^2) total.

def cumulative_sum_brute(numbers):
    return [sum(numbers[:i + 1]) for i in range(len(numbers))]

print(cumulative_sum_brute([1, 2, 3, 4]))   # [1, 3, 6, 10]

**Optimal:** Carry one running total forward instead of re-summing — the pseudocode's exact shape. O(n) time, since each number is only ever added once.

In [ ]:
from itertools import accumulate

# Optimal — carry a running total. O(n) time.

def cumulative_sum_optimal(numbers):
    result = []
    running_total = 0
    for n in numbers:
        running_total += n
        result.append(running_total)
    return result

print(cumulative_sum_optimal([1, 2, 3, 4]))   # [1, 3, 6, 10]

# itertools.accumulate does exactly this, and returns an iterator --
# wrap it in list() to materialize it. It also accepts a different
# combining function (e.g. max, for a running maximum) via its `func`
# argument, generalizing well past addition.
def cumulative_sum_accumulate(numbers):
    return list(accumulate(numbers))

print(cumulative_sum_accumulate([1, 2, 3, 4]))   # [1, 3, 6, 10]

In [ ]:
# Tests

assert cumulative_sum_optimal([]) == []                              # edge: empty list
assert cumulative_sum_optimal([5]) == [5]                             # edge: single element
assert cumulative_sum_optimal([1, 2, 3, 4]) == [1, 3, 6, 10]
assert cumulative_sum_optimal([0, 0, 0]) == [0, 0, 0]                 # edge: all zeros
assert cumulative_sum_optimal([-1, 1, -1, 1]) == [-1, 0, -1, 0]        # negative numbers, running total can dip and recover
assert cumulative_sum_optimal([5, -10, 3]) == [5, -5, -2]              # running total goes negative

for numbers in ([], [5], [1, 2, 3, 4], [0, 0, 0], [-1, 1, -1, 1], [5, -10, 3]):
    assert (
        cumulative_sum_brute(numbers)
        == cumulative_sum_optimal(numbers)
        == cumulative_sum_accumulate(numbers)
    )

print("All Q28 tests passed.")

**Interview traps:**
- **`[sum(numbers[:i+1]) for i in range(len(numbers))]` looks like a clean one-liner but is quietly O(n²)** — each `sum(numbers[:i+1])` re-adds every earlier number from scratch. This is the same "elegant but repeats work" shape as Q27's `sum(lists, [])`; both look declarative and hide a nested cost inside a single line.
- **`result[0]` is `numbers[0]`, not `0`.** A common off-by-one is initializing the output with a leading `0` (as if index 0 meant "sum of nothing yet") — but the question defines the cumulative sum *up to and including* each index, so the very first output element already includes the first input number.
- **`itertools.accumulate` returns an iterator, not a list** — same lazy-evaluation trap as `chain.from_iterable` in Q27; wrap it in `list(...)` if you need to index into it or check its length rather than just iterate once.
- **`accumulate` generalizes past summing** — pass a different two-argument function (`max`, `min`, a custom lambda) via its second parameter to get a running maximum, running minimum, or any other "carry forward and combine" computation. Knowing this shows you understand the *shape* of the problem (fold left with a running value), not just the specific case of addition.

---
## Q29. Leap Year Check

> **Leap Year Check:** Determine if a given year is a leap year. (Leap years are divisible by 4, except century years not divisible by 400.)
> _Hint:_ Check the divisibility conditions: `year % 4 == 0` and (`year % 100 != 0` or `year % 400 == 0`).

**Restate:** Given a year, decide whether it's a leap year under the Gregorian rule: divisible by 4, *unless* it's also divisible by 100 — in which case it's only still a leap year if it's *also* divisible by 400.

**Algorithmic thinking (no syntax):**
1. If the year isn't divisible by 4, it's definitely not a leap year — stop.
2. If it *is* divisible by 4, but it's also a century year (divisible by 100), that's usually a disqualifier...
3. ...*unless* it's also divisible by 400, which overrides the century disqualification and makes it a leap year after all.
4. Otherwise (divisible by 4, not a century year), it's a leap year.

This has no brute-force-vs-optimal complexity split at all — it's three divisibility checks, O(1), no matter how you write it. What actually varies between a bad and a good answer here is *correctness under the exception-to-the-exception rule* (step 3), which is exactly what trips people up.

In [ ]:
# "Brute force" here is actually a demonstration of the classic WRONG
# answer: checking only divisibility by 4, and forgetting the century
# exception entirely. It's correct for MOST years, which is exactly
# what makes the bug dangerous -- it passes casual testing.

def is_leap_year_wrong(year):
    return year % 4 == 0   # missing the century rule

print(is_leap_year_wrong(2024))   # True -- correct, by luck
print(is_leap_year_wrong(1900))   # True -- WRONG: 1900 was not a leap year

**Optimal (and correct):** All three conditions from the pseudocode, combined into one boolean expression. Still O(1) — the point isn't speed, it's making sure the century exception and its own exception are both actually encoded.

In [ ]:
import calendar

# Optimal — full three-part rule. O(1).

def is_leap_year_optimal(year):
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)

print(is_leap_year_optimal(2024))   # True
print(is_leap_year_optimal(1900))   # False -- century year, not divisible by 400
print(is_leap_year_optimal(2000))   # True -- century year, but divisible by 400

# The real-world answer: Python's standard library already has this.
print(calendar.isleap(1900))   # False

In [ ]:
# Tests

assert is_leap_year_optimal(2024) is True                # divisible by 4, not a century year
assert is_leap_year_optimal(2023) is False                # not divisible by 4
assert is_leap_year_optimal(1900) is False                # edge: century year, NOT divisible by 400
assert is_leap_year_optimal(2000) is True                 # edge: century year, divisible by 400
assert is_leap_year_optimal(2100) is False                # edge: future century year, not divisible by 400
assert is_leap_year_optimal(1) is False                    # edge: small year
assert is_leap_year_optimal(4) is True                     # edge: smallest positive leap year
assert is_leap_year_optimal(0) is True                     # edge: year 0, divisible by 4, 100, and 400

for year in list(range(1800, 2200)):
    assert is_leap_year_optimal(year) == calendar.isleap(year), year

# Confirm the "wrong" version really does disagree with the correct one
# on the exact case the century rule exists for.
assert is_leap_year_wrong(1900) != is_leap_year_optimal(1900)

print("All Q29 tests passed.")

**Interview traps:**
- **Forgetting the century exception entirely is the single most common mistake on this question**, and it's dangerous precisely because it's *right 96% of the time* — every non-century year divisible by 4 gets the correct answer. Only century years (1700, 1800, 1900, 2100, ...) expose the bug, so a test suite that doesn't specifically include one will pass anyway.
- **1900 not leap, 2000 leap — this exact pair is the canonical test case for this question.** If you don't test at least one century-divisible-by-100-but-not-400 year and one divisible-by-400 year, you haven't actually verified the part of the logic that makes this question interesting.
- **Operator precedence: `and` binds tighter than `or` in Python** (as in most languages), so `year % 4 == 0 and year % 100 != 0 or year % 400 == 0` parses as `(year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)` — which happens to produce the same *results* as the explicitly-parenthesized version in the pseudocode for this particular rule, but relying on precedence you haven't verified is a habit that bites you on a different boolean expression eventually. Parenthesize explicitly and know why the parens are or aren't load-bearing.
- **`calendar.isleap()` is the real answer for real code** — same pattern as `math.gcd`, `math.factorial`: know the manual logic well enough to explain it, but don't hand-rewrite a standard-library one-liner in production.
- **Negative and year-0 inputs are a genuine edge case for a *proleptic* Gregorian calendar** (extending the modern calendar backward past its historical introduction in 1582) — mathematically well-defined by the same formula, but worth flagging as an assumption if the question is about real historical calendars rather than pure arithmetic.

---
## Q30. Transpose Matrix

> **Transpose Matrix:** Compute the transpose of a given 2D matrix (rows become columns and vice versa).
> _Hint:_ Use nested loops or `zip(*matrix)` in Python to unpack rows as columns.

**Restate:** Given a 2D matrix (a list of equal-length rows), produce a new matrix where the element at row `i`, column `j` moves to row `j`, column `i` — rows become columns, columns become rows. We'll assume the input is *rectangular* (every row the same length); a "matrix" with jagged row lengths isn't really a matrix, and we'll revisit what happens if that assumption is violated in the traps.

**Algorithmic thinking (no syntax):**
1. For each column position in the original matrix, start building a new row.
2. To build that new row, walk down every row of the original matrix, picking out the element that was sitting at the column position you're currently working on.
3. Once you've picked one element from every original row, that's a complete new row — add it to the result.
4. Repeat for every column position in the original matrix.

This is a pure re-indexing operation — every element moves exactly once, no element is read or written more than once — so there's no repeated-work story here the way there was in Q27/Q28. What's actually worth attention is how you *build* the result without accidentally sharing structure between rows, which is where the real trap lives.

In [ ]:
# Brute force — manual nested loop, re-indexing explicitly.
# O(rows * cols) time and space; this is genuinely the minimum
# necessary work, since every element has to move somewhere.

def transpose_brute(matrix):
    if not matrix:
        return []
    num_rows, num_cols = len(matrix), len(matrix[0])
    result = []
    for col in range(num_cols):
        new_row = []
        for row in range(num_rows):
            new_row.append(matrix[row][col])
        result.append(new_row)
    return result

print(transpose_brute([[1, 2, 3], [4, 5, 6]]))
# [[1, 4], [2, 5], [3, 6]]

**Optimal:** `zip(*matrix)` — same O(rows·cols) time (there's no faster asymptotic option; every element genuinely has to move), but it eliminates the manual double-indexing entirely, which is exactly where off-by-one bugs like to hide. `zip(*matrix)` unpacks each row as a separate argument to `zip`, which then pairs up the *i*-th element of every row into a tuple — that's a transpose, expressed directly rather than simulated with index arithmetic.

In [ ]:
# Optimal — zip(*matrix). O(rows * cols) time, no manual indexing.

def transpose_optimal(matrix):
    return [list(row) for row in zip(*matrix)]   # zip yields tuples; list() per row for a plain list-of-lists

print(transpose_optimal([[1, 2, 3], [4, 5, 6]]))
# [[1, 4], [2, 5], [3, 6]]

In [ ]:
# The classic gotcha, demonstrated: pre-allocating a result matrix with
# [[0] * cols] * rows does NOT create `rows` independent lists. It
# creates one list, repeated `rows` times BY REFERENCE -- every "row"
# is the exact same list object in memory.

shared_rows = [[0] * 3] * 2      # looks like a 2x3 grid of independent zeros
print(shared_rows)                # [[0, 0, 0], [0, 0, 0]] -- looks fine so far

shared_rows[0][0] = 99            # mutate what looks like just the first row
print(shared_rows)                # [[99, 0, 0], [99, 0, 0]] -- BOTH rows changed!

# The fix: build each row with its own list comprehension, so each
# row is a genuinely separate list object.
independent_rows = [[0] * 3 for _ in range(2)]
independent_rows[0][0] = 99
print(independent_rows)           # [[99, 0, 0], [0, 0, 0]] -- only the first row changed, as intended

In [ ]:
# Tests

assert transpose_optimal([]) == []                                              # edge: empty matrix
assert transpose_optimal([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]     # rectangular, not square
assert transpose_optimal([[1, 2], [3, 4]]) == [[1, 3], [2, 4]]                   # square matrix
assert transpose_optimal([[1, 2, 3]]) == [[1], [2], [3]]                        # edge: single row
assert transpose_optimal([[1], [2], [3]]) == [[1, 2, 3]]                        # edge: single column
assert transpose_optimal(transpose_optimal([[1, 2], [3, 4], [5, 6]])) == [[1, 2], [3, 4], [5, 6]]  # transposing twice restores the original

for matrix in ([[1, 2, 3], [4, 5, 6]], [[1, 2], [3, 4]], [[1, 2, 3]], [[1], [2], [3]]):
    assert transpose_brute(matrix) == transpose_optimal(matrix)

print("All Q30 tests passed.")

In [ ]:
# A ragged (non-rectangular) input -- rows of different lengths --
# is technically not a valid matrix, but it's worth seeing how each
# approach fails, since they fail differently.

ragged = [[1, 2, 3], [4, 5]]   # second row is missing an element

print(transpose_optimal(ragged))   # [[1, 4], [2, 5]] -- silently drops column 3 entirely, no error

try:
    transpose_brute(ragged)
    print("transpose_brute did not raise")
except IndexError as e:
    print(f"transpose_brute raised: {e}")   # crashes explicitly instead of dropping data

**Interview traps:**
- **`[[0] * cols] * rows` is one of the single most common real-world Python bugs, not just an interview trap.** Multiplying a list *of mutable objects* doesn't copy the objects — it copies the *reference* to the same object, `rows` times. Mutating "one row" mutates all of them simultaneously, because there's only actually one row. The fix is always some form of building each row independently (a list comprehension with its own `range`, or `copy.deepcopy`), never list multiplication, whenever the elements themselves are mutable.
- **`zip(*matrix)` silently truncates to the shortest row on ragged input — it does not raise an error.** The manual nested-loop version *does* raise `IndexError` the moment it walks off the end of a short row. Neither behavior is "more correct" in the abstract; but they're different, and if the input isn't guaranteed rectangular, you need to decide (and state) which failure mode you want rather than being surprised by whichever one your code happens to produce.
- **`zip(*matrix)` yields tuples, not lists.** If downstream code needs to mutate rows of the result (`result[0][0] = 5`), you need the `[list(row) for row in zip(*matrix)]` wrapping — a bare `list(zip(*matrix))` gives you a list of *tuples*, which are immutable.
- **`*matrix` unpacking requires each row to be a separate argument, which is exactly what `*` does for `zip`** — worth being able to explain `zip(*matrix)` isn't a special "transpose operator," it's the ordinary argument-unpacking operator applied to a function (`zip`) that happens to produce a transpose when given rows as separate arguments. Understanding *why* it works, not just that it's the one-liner to reach for, is what separates "I memorized this" from "I understand this."